<a href="https://colab.research.google.com/github/sebasarellano/Aprendizaje_Automatico/blob/main/Preprocesamiento_y_Particion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Selección, partición y preprocesamiento de datos

## Objetivo

Documentar la fuente y las características del conjunto de datos seleccionado, proponer una partición para aprendizaje supervisado y justificar las transformaciones necesarias para preparar los datos.

Se utilizará el conjunto de datos *Predict Students' Dropout and Academic Success*, publicado en UCI Machine Learning Repository.


## 1. Fuente y descripción del conjunto de datos

**Nombre:** Predict Students' Dropout and Academic Success.

**Fuente:** [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/697/predict+students+dropout+and+academic+success).

**Número de registros:** 4.424.

**Variables predictoras:** 36.

**Variable objetivo:** `Target`, que identifica tres resultados académicos: abandono (*Dropout*), continuidad de los estudios (*Enrolled*) y graduación (*Graduate*).

**Unidad de observación:** cada registro representa a un estudiante de una institución de educación superior.

**Descripción:** el conjunto contiene información relacionada con la admisión, las características demográficas y socioeconómicas, y el rendimiento académico de estudiantes de diferentes programas educativos. Puede utilizarse para estudiar la clasificación de resultados académicos y explorar agrupaciones de estudiantes según sus características.

**Tipos de variables:** incluye variables numéricas continuas, enteras y categóricas.



## 2. Acceso y carga de los datos

Los datos se obtienen directamente desde UCI Machine Learning Repository mediante la biblioteca `ucimlrepo`. De esta manera, el repositorio de GitHub contendrá el notebook y las instrucciones de acceso, sin incorporar una copia innecesaria del archivo original.

Se conservarán por separado las variables predictoras (`X`) y la variable objetivo (`y`). Los datos originales no se modificarán durante la carga.


In [3]:
pip install ucimlrepo

In [4]:

from ucimlrepo import fetch_ucirepo

# Obtener el conjunto de datos desde UCI
dataset = fetch_ucirepo(id=697)

# Separar las variables predictoras y la variable objetivo
X = dataset.data.features.copy()
y = dataset.data.targets.copy()

# Mostrar las dimensiones originales
print("Número de registros:", X.shape[0])
print("Número de variables predictoras:", X.shape[1])
print("Número de variables objetivo:", y.shape[1])


Número de registros: 4424
Número de variables predictoras: 36
Número de variables objetivo: 1



## 3. Propuesta de partición de los datos

Para el flujo de aprendizaje supervisado se propone dividir los datos en dos conjuntos:

- **Entrenamiento (80 %):** se utilizará para ajustar las transformaciones de preprocesamiento y, posteriormente, entrenar los modelos.
- **Prueba (20 %):** se reservará para evaluar el desempeño final de los modelos con datos que no participaron en el entrenamiento.

Se utilizará una partición estratificada según la variable objetivo `Target`, con el propósito de conservar aproximadamente la proporción de cada clase en ambos conjuntos. Se fijará una semilla aleatoria para que la división sea reproducible.


In [5]:

from sklearn.model_selection import train_test_split

# Dividir los datos en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=2026,
    stratify=y
)

# Mostrar el tamaño de cada partición
print("Registros de entrenamiento:", X_train.shape[0])
print("Registros de prueba:", X_test.shape[0])

print("Porcentaje de entrenamiento:",
      round(len(X_train) / len(X) * 100, 2), "%")

print("Porcentaje de prueba:",
      round(len(X_test) / len(X) * 100, 2), "%")


Registros de entrenamiento: 3539
Registros de prueba: 885
Porcentaje de entrenamiento: 80.0 %
Porcentaje de prueba: 20.0 %



## 4. Transformaciones seleccionadas

Se utilizarán dos transformaciones:

- **Estandarización (Z-score):** se aplicará a las variables numéricas para expresar sus valores en función de la media y la desviación estándar del conjunto de entrenamiento.
- **Codificación One-Hot:** se aplicará a las variables categóricas para representarlas sin establecer un orden artificial entre sus categorías.

No se aplicarán normalización Min-Max ni discretización, ya que no son necesarias para esta propuesta inicial.

Las transformaciones se ajustarán exclusivamente con los datos de entrenamiento. Después, se aplicarán a los datos de prueba utilizando los parámetros aprendidos durante el entrenamiento, sin volver a ajustarlos.





In [8]:
# Nombres de las variables
for columna in X.columns:
    print(repr(columna))

'Marital Status'
'Application mode'
'Application order'
'Course'
'Daytime/evening attendance'
'Previous qualification'
'Previous qualification (grade)'
'Nacionality'
"Mother's qualification"
"Father's qualification"
"Mother's occupation"
"Father's occupation"
'Admission grade'
'Displaced'
'Educational special needs'
'Debtor'
'Tuition fees up to date'
'Gender'
'Scholarship holder'
'Age at enrollment'
'International'
'Curricular units 1st sem (credited)'
'Curricular units 1st sem (enrolled)'
'Curricular units 1st sem (evaluations)'
'Curricular units 1st sem (approved)'
'Curricular units 1st sem (grade)'
'Curricular units 1st sem (without evaluations)'
'Curricular units 2nd sem (credited)'
'Curricular units 2nd sem (enrolled)'
'Curricular units 2nd sem (evaluations)'
'Curricular units 2nd sem (approved)'
'Curricular units 2nd sem (grade)'
'Curricular units 2nd sem (without evaluations)'
'Unemployment rate'
'Inflation rate'
'GDP'


In [9]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Identificar las variables categóricas
variables_categoricas = [
    "Marital Status",
    "Application mode",
    "Application order",
    "Course",
    "Daytime/evening attendance",
    "Previous qualification",
    "Nacionality",
    "Mother's qualification",
    "Father's qualification",
    "Mother's occupation",
    "Father's occupation",
    "Displaced",
    "Educational special needs",
    "Debtor",
    "Tuition fees up to date",
    "Gender",
    "Scholarship holder",
    "International"
]

# Identificar las variables numéricas restantes
variables_numericas = [
    columna for columna in X.columns
    if columna not in variables_categoricas
]

# Estandarizar las variables numéricas
escalador = StandardScaler()

X_train_numericas = pd.DataFrame(
    escalador.fit_transform(X_train[variables_numericas]),
    columns=variables_numericas,
    index=X_train.index
)

X_test_numericas = pd.DataFrame(
    escalador.transform(X_test[variables_numericas]),
    columns=variables_numericas,
    index=X_test.index
)

# Codificar las variables categóricas con One-Hot
codificador = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

X_train_categoricas = pd.DataFrame(
    codificador.fit_transform(X_train[variables_categoricas]),
    columns=codificador.get_feature_names_out(variables_categoricas),
    index=X_train.index
)

X_test_categoricas = pd.DataFrame(
    codificador.transform(X_test[variables_categoricas]),
    columns=codificador.get_feature_names_out(variables_categoricas),
    index=X_test.index
)

# Unir las variables numéricas y categóricas transformadas
X_train_preparado = pd.concat(
    [X_train_numericas, X_train_categoricas],
    axis=1
)

X_test_preparado = pd.concat(
    [X_test_numericas, X_test_categoricas],
    axis=1
)

# Comprobar las dimensiones
print("Entrenamiento original:", X_train.shape)
print("Prueba original:", X_test.shape)

print("Entrenamiento preprocesado:", X_train_preparado.shape)
print("Prueba preprocesada:", X_test_preparado.shape)

Entrenamiento original: (3539, 36)
Prueba original: (885, 36)
Entrenamiento preprocesado: (3539, 251)
Prueba preprocesada: (885, 251)



## 5. Resultados y organización de los datos

La partición generó 3.539 registros de entrenamiento y 885 registros de prueba, correspondientes aproximadamente al 80 % y al 20 % del conjunto original.

Después del preprocesamiento, cada partición conserva sus registros y contiene 251 columnas. El aumento de columnas se debe a la codificación One-Hot de las variables categóricas.

Se mantienen diferenciados los datos originales, las particiones de entrenamiento y prueba, y los datos transformados. La estandarización y la codificación One-Hot se ajustaron únicamente con entrenamiento y se aplicaron a prueba utilizando los parámetros vistos en clase.

